# 📊 Evaluasi Model — Confusion Matrix
### Analisis Akurasi Model Deteksi Penyakit Jagung
> **Jalankan notebook ini di Google Colab yang sama** di mana Anda sudah training model.
> Jika session baru, upload ulang dataset dan file `best.pt`

## ✅ Step 1: Install Library

In [ ]:
!pip install ultralytics seaborn scikit-learn -q
print('✅ Library berhasil diinstall!')

## ✅ Step 2: Cek Apakah Model & Dataset Masih Ada
Jika session Colab masih sama dengan saat training, file akan ada. Jika tidak, lanjut ke Step 2b.

In [ ]:
import os

model_path = '/content/runs/jagung/weights/best.pt'
yaml_path_candidates = [
    '/content/dataset/data.yaml',
    '/content/dataset/data/data.yaml',
]

yaml_path = None
for yp in yaml_path_candidates:
    if os.path.exists(yp):
        yaml_path = yp
        break

# Cari secara otomatis jika tidak ditemukan
if not yaml_path:
    import glob
    found = glob.glob('/content/dataset/**/*.yaml', recursive=True)
    if found:
        yaml_path = found[0]

print(f'Model  : {model_path} → {"✅ Ada" if os.path.exists(model_path) else "❌ Tidak ada"}')
print(f'Dataset: {yaml_path} → {"✅ Ada" if yaml_path and os.path.exists(yaml_path) else "❌ Tidak ada"}')

if not os.path.exists(model_path) or not yaml_path:
    print('\n⚠️ File tidak ditemukan! Jalankan Step 2b di bawah.')
else:
    print('\n🎉 Semua file tersedia! Langsung ke Step 3.')

## ⚡ Step 2b (Opsional): Upload File Jika Session Baru
Jalankan cell ini **hanya jika** Step 2 menunjukkan file tidak ada.

In [ ]:
import zipfile
from google.colab import files

# --- Upload Dataset ---
print('📂 Upload file ZIP dataset Anda...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')
print('✅ Dataset diekstrak!')

# Cari data.yaml
import glob
found = glob.glob('/content/dataset/**/*.yaml', recursive=True)
if found:
    yaml_path = found[0]
    print(f'📄 data.yaml: {yaml_path}')

    # Fix path
    import yaml
    dataset_root = os.path.dirname(yaml_path)
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    data['train'] = os.path.join(dataset_root, 'train', 'images')
    data['val']   = os.path.join(dataset_root, 'valid', 'images')
    data['test']  = os.path.join(dataset_root, 'test', 'images')
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)
    print('✅ Path diperbarui!')

# --- Upload best.pt ---
print('\n📂 Upload file best.pt Anda...')
uploaded2 = files.upload()
pt_name = list(uploaded2.keys())[0]
os.makedirs('/content/runs/jagung/weights', exist_ok=True)
os.rename(pt_name, '/content/runs/jagung/weights/best.pt')
model_path = '/content/runs/jagung/weights/best.pt'
print(f'✅ Model disimpan di: {model_path}')

## ✅ Step 3: Jalankan Validasi & Generate Confusion Matrix (YOLOv8 Built-in)

In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO(model_path)

# Validasi pada data test
results = model.val(
    data=yaml_path,
    split='test',        # Gunakan data test
    project='/content/eval',
    name='confusion_matrix',
    exist_ok=True,
    plots=True,          # Generate semua plot termasuk confusion matrix
    save_json=True,
)

print('\n✅ Validasi selesai!')
print(f'\n📊 Metrik Keseluruhan:')
print(f'  Precision   : {results.box.mp:.4f}')
print(f'  Recall      : {results.box.mr:.4f}')
print(f'  mAP@50      : {results.box.map50:.4f}')
print(f'  mAP@50-95   : {results.box.map:.4f}')

## ✅ Step 4: Tampilkan Confusion Matrix (YOLOv8 Built-in)

In [ ]:
from IPython.display import Image, display
import os

cm_path = '/content/eval/confusion_matrix/confusion_matrix.png'
cm_norm_path = '/content/eval/confusion_matrix/confusion_matrix_normalized.png'

if os.path.exists(cm_path):
    print('📊 Confusion Matrix:')
    display(Image(filename=cm_path, width=700))
else:
    print('❌ Confusion matrix tidak ditemukan di path default.')

print('\n')

if os.path.exists(cm_norm_path):
    print('📊 Confusion Matrix (Normalized):')
    display(Image(filename=cm_norm_path, width=700))
else:
    print('ℹ️ Normalized confusion matrix tidak tersedia.')

## ✅ Step 5: Confusion Matrix Custom (Lebih Detail dengan Scikit-learn)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import yaml
import os
from pathlib import Path

# Load data.yaml
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

class_names = list(data_cfg['names'].values()) if isinstance(data_cfg['names'], dict) else data_cfg['names']
print(f'Kelas: {class_names}')

# Load model
model = YOLO(model_path)

# Get test images
test_img_dir = data_cfg.get('test', os.path.join(os.path.dirname(yaml_path), 'test', 'images'))
test_label_dir = test_img_dir.replace('images', 'labels')

print(f'Test images : {test_img_dir}')
print(f'Test labels : {test_label_dir}')

# Collect predictions vs ground truth (image-level classification)
y_true = []
y_pred = []

img_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.webp']
test_images = sorted([f for f in Path(test_img_dir).iterdir() if f.suffix.lower() in img_extensions])

print(f'\nTotal test images: {len(test_images)}')
print('Memproses...')

for img_path in test_images:
    # Get ground truth label
    label_path = Path(test_label_dir) / (img_path.stem + '.txt')
    
    if not label_path.exists():
        continue
    
    # Read ground truth (use the class with most instances, or first)
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    if not lines:
        continue
    
    # Get all GT classes in this image
    gt_classes = [int(line.strip().split()[0]) for line in lines if line.strip()]
    
    # Run prediction
    results = model.predict(str(img_path), verbose=False, conf=0.35)
    
    pred_classes = []
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        pred_classes = results[0].boxes.cls.cpu().numpy().astype(int).tolist()
    
    # Match each GT with best prediction
    for gt_cls in gt_classes:
        if gt_cls in pred_classes:
            y_true.append(gt_cls)
            y_pred.append(gt_cls)
            pred_classes.remove(gt_cls)  # Remove matched
        elif pred_classes:
            # Take highest confidence wrong prediction
            y_true.append(gt_cls)
            y_pred.append(pred_classes[0])
            pred_classes.pop(0)
        else:
            # Missed detection (false negative)
            y_true.append(gt_cls)
            y_pred.append(-1)  # missed

print(f'\n✅ Total deteksi yang dievaluasi: {len(y_true)}')

## ✅ Step 6: Visualisasi Confusion Matrix (Custom Seaborn)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Filter out missed detections for confusion matrix
y_true_filtered = []
y_pred_filtered = []
missed = 0

for gt, pred in zip(y_true, y_pred):
    if pred == -1:
        missed += 1
    else:
        y_true_filtered.append(gt)
        y_pred_filtered.append(pred)

print(f'Total deteksi  : {len(y_true)}')
print(f'Missed (FN)    : {missed}')
print(f'Detected       : {len(y_true_filtered)}')

# Confusion Matrix
cm = confusion_matrix(y_true_filtered, y_pred_filtered)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# ========== Plot 1: Confusion Matrix (Counts) ==========
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0],
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'shrink': 0.8}
)
axes[0].set_xlabel('Prediksi', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Aktual (Ground Truth)', fontsize=12, fontweight='bold')
axes[0].set_title('Confusion Matrix\n(Jumlah Deteksi)', fontsize=14, fontweight='bold', pad=15)
axes[0].tick_params(axis='both', which='major', labelsize=10)

# Normalized (percentage)
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.1%',
    cmap='YlGn',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[1],
    linewidths=0.5,
    linecolor='white',
    vmin=0,
    vmax=1,
    cbar_kws={'shrink': 0.8}
)
axes[1].set_xlabel('Prediksi', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Aktual (Ground Truth)', fontsize=12, fontweight='bold')
axes[1].set_title('Confusion Matrix\n(Persentase / Normalized)', fontsize=14, fontweight='bold', pad=15)
axes[1].tick_params(axis='both', which='major', labelsize=10)

plt.tight_layout(pad=3)
plt.savefig('/content/confusion_matrix_custom.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Confusion Matrix tersimpan di /content/confusion_matrix_custom.png')

## ✅ Step 7: Classification Report (Precision, Recall, F1-Score)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Classification Report
print('=' * 60)
print('📋 CLASSIFICATION REPORT')
print('=' * 60)
print()
print(classification_report(
    y_true_filtered,
    y_pred_filtered,
    target_names=class_names,
    digits=4
))

# Overall Accuracy
acc = accuracy_score(y_true_filtered, y_pred_filtered)
print(f'\n🎯 Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'📊 Detection Rate  : {len(y_true_filtered)}/{len(y_true)} ({len(y_true_filtered)/len(y_true)*100:.1f}%)')
print(f'❌ Missed (FN)      : {missed}/{len(y_true)} ({missed/len(y_true)*100:.1f}%)')

## ✅ Step 8: Visualisasi Per Kelas (Bar Chart)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import numpy as np

precision_per_class = precision_score(y_true_filtered, y_pred_filtered, average=None, zero_division=0)
recall_per_class = recall_score(y_true_filtered, y_pred_filtered, average=None, zero_division=0)
f1_per_class = f1_score(y_true_filtered, y_pred_filtered, average=None, zero_division=0)

x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2d6a4f', '#40916c', '#95d5b2']

bars1 = ax.bar(x - width, precision_per_class, width, label='Precision', color=colors[0], edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x,         recall_per_class,    width, label='Recall',    color=colors[1], edgecolor='white', linewidth=0.5)
bars3 = ax.bar(x + width, f1_per_class,        width, label='F1-Score',  color=colors[2], edgecolor='white', linewidth=0.5)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 4),
                    textcoords='offset points',
                    ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

ax.set_xlabel('Kelas Penyakit', fontsize=12, fontweight='bold')
ax.set_ylabel('Skor', fontsize=12, fontweight='bold')
ax.set_title('Precision, Recall, dan F1-Score per Kelas', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(class_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/content/metrics_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Chart tersimpan di /content/metrics_per_class.png')

## ✅ Step 9: Download Semua Hasil Evaluasi

In [ ]:
import shutil
from google.colab import files

# Buat folder hasil
eval_dir = '/content/hasil_evaluasi'
os.makedirs(eval_dir, exist_ok=True)

# Copy file evaluasi
files_to_copy = [
    ('/content/confusion_matrix_custom.png', 'confusion_matrix_custom.png'),
    ('/content/metrics_per_class.png', 'metrics_per_class.png'),
]

# Copy YOLOv8 built-in plots
yolo_plots = [
    '/content/eval/confusion_matrix/confusion_matrix.png',
    '/content/eval/confusion_matrix/confusion_matrix_normalized.png',
    '/content/eval/confusion_matrix/P_curve.png',
    '/content/eval/confusion_matrix/R_curve.png',
    '/content/eval/confusion_matrix/F1_curve.png',
    '/content/eval/confusion_matrix/PR_curve.png',
]

for src in yolo_plots:
    if os.path.exists(src):
        files_to_copy.append((src, os.path.basename(src)))

for src, dst in files_to_copy:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(eval_dir, dst))
        print(f'  ✅ {dst}')

# Save classification report as text
report = classification_report(y_true_filtered, y_pred_filtered, target_names=class_names, digits=4)
with open(os.path.join(eval_dir, 'classification_report.txt'), 'w') as f:
    f.write('CLASSIFICATION REPORT\n')
    f.write('=' * 60 + '\n\n')
    f.write(report)
    f.write(f'\nOverall Accuracy: {accuracy_score(y_true_filtered, y_pred_filtered):.4f}')
print('  ✅ classification_report.txt')

# Zip dan download
shutil.make_archive('/content/hasil_evaluasi', 'zip', eval_dir)
print('\n📥 Downloading...')
files.download('/content/hasil_evaluasi.zip')
print('\n✅ Selesai! Cek folder Downloads Anda.')

## 🎉 Selesai!

### Ringkasan file yang didownload:
| File | Deskripsi |
|------|-----------|
| `confusion_matrix.png` | Confusion Matrix dari YOLOv8 |
| `confusion_matrix_normalized.png` | Confusion Matrix (normalized) dari YOLOv8 |
| `confusion_matrix_custom.png` | Custom Confusion Matrix (counts + percentage) |
| `metrics_per_class.png` | Bar chart Precision/Recall/F1 per kelas |
| `P_curve.png` | Precision curve |
| `R_curve.png` | Recall curve |
| `F1_curve.png` | F1-Score curve |
| `PR_curve.png` | Precision-Recall curve |
| `classification_report.txt` | Laporan lengkap per kelas |